# Introduction to Strands Agents with MCP

This notebook introduces the **Model Context Protocol (MCP)** — how a Strands Agent connects to an MCP server, discovers tools, and inspects a Neo4j knowledge graph.

**Learning Objectives:**
- Connect a Strands Agent to an MCP server via Streamable HTTP transport
- Discover tools automatically via `list_tools_sync()`
- Inspect the graph schema using the `get-schema` tool
- Understand MCP's three-component architecture: Agent → MCP Server → Data Source

**How it works:**
1. `MCPClient` connects to the Neo4j MCP Server via an AWS AgentCore Gateway
2. `list_tools_sync()` discovers available tools (`get-schema`, `read-cypher`)
3. The discovered tools are passed directly to the Strands `Agent`
4. The agent calls tools based on your questions — no custom wrappers needed

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

## 1. Configuration and Imports

In [ ]:
import os

from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("\nConfiguration loaded!")

## 2. Initialize Model and MCP Connection

- **BedrockModel**: Configures Claude on Amazon Bedrock with temperature 0 for deterministic responses
- **MCPClient**: Connects to the Neo4j MCP Server and discovers available tools. `list_tools_sync()` returns tool wrappers that the agent can call directly — this is the standard Strands pattern for MCP integration.

In [ ]:
from botocore.config import Config as BotocoreConfig

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools from the MCP server
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")
print(f"\nModel: {MODEL_ID}")

## 3. Create the Agent

Pass the discovered MCP tools directly to the agent. The agent inspects each tool's name, description, and parameter schema to decide when and how to call it — no `@tool` wrappers or manual tool selection needed.

The system prompt instructs the agent to follow a **schema-first approach**: always retrieve the database schema before writing Cypher. This grounds the agent's queries in the actual graph structure.

In [ ]:
SYSTEM_PROMPT = """You are a financial analysis assistant with access to a Neo4j knowledge graph containing SEC 10-K filing data (companies, products, risk factors, asset managers, and document chunks).

Rules:
1. Always retrieve the database schema before writing Cypher queries.
2. Only use read-only Cypher (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
3. Use modern Cypher syntax:
   - Use elementId(n) instead of id(n)
   - Use COUNT{ pattern } instead of size((pattern))
   - Use EXISTS{ pattern } instead of exists((pattern))
   - Always use $parameter syntax for dynamic values
4. Keep results concise — limit to 10 rows unless asked otherwise.
"""

agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=mcp_tools,
)


def query(question: str):
    """Ask the agent a question about the knowledge graph."""
    print(f'Question: "{question}"')
    print("-" * 60)
    response = agent(question)
    print(f"\n{response}")
    return response


print("Agent ready!")

## 4. Query the Knowledge Graph

Ask the agent to retrieve the graph schema. Watch the tool calls in the output — the agent calls `get-schema` to learn the graph structure and summarizes what it finds.

In [ ]:
# Schema discovery — the agent calls get-schema and summarizes what it finds
result = query("What is the database schema? Give me a brief summary of the node labels, relationship types, and key properties.")

In [ ]:
# Simple query — the agent writes a Cypher query and executes it via read-cypher
result = query("How many companies are in the database?")

## Summary

You connected a Strands Agent to a Neo4j MCP Server and inspected the knowledge graph:

| Component | Role |
|-----------|------|
| **Strands Agent** | Receives your question, decides which tools to call, interprets results |
| **MCPClient** | Discovers tools from the MCP server and provides them to the agent |
| **Neo4j MCP Server** | Exposes `get-schema` and `read-cypher` tools over MCP |
| **BedrockModel** | Powers the agent's reasoning with Claude on Amazon Bedrock |

The agent discovered tools automatically and used them to retrieve the schema and run a simple query. In the next notebook, you'll build **`@tool` wrappers** that combine vector search with graph traversal — the **Cypher Templates** pattern for reliable, pre-written queries.

---

**Next:** [Graph-Enriched Search](02_graph_enriched_search.ipynb) — pre-written Cypher templates with vector search and graph traversal